In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
#downloading
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
])

train_dataset = datasets.STL10(root="../data", split='train', download=True, transform=transform)
test_dataset = datasets.STL10(root="../data", split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100.0%


In [12]:
model = models.resnet18(pretrained=True)
model = model.to(device)

g:\DL-Assignment-3\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\DL-Assignment-3\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
for param in model.parameters():
    param.requires_grad = False

In [14]:
#replacing our final layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)  

model.fc.requires_grad_(True) 

Linear(in_features=512, out_features=10, bias=True)

In [15]:
#loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [16]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [17]:
#eval funtion
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
#training execution
#as it was taking time more than an hour for me , I have stopped at epoch 3 isntead of 5
epochs = 3

for epoch in range(epochs):
    loss = train(model, train_loader, optimizer, criterion)
    acc = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Test Accuracy = {acc:.2f}%")

Epoch 1: Loss = 0.9279, Test Accuracy = 91.88%
Epoch 2: Loss = 0.3279, Test Accuracy = 93.44%
Epoch 3: Loss = 0.2452, Test Accuracy = 93.78%


KeyboardInterrupt: 

In [19]:
torch.save(model.state_dict(), "../models/resnet18_stl10.pth")

with open("../results/task2_accuracy.txt", "w") as f:
    f.write(f"Final Test Accuracy: {acc:.2f}%")